# Lesson 2 - Vector Search - Code Along 1

In [1]:
# dependencies
import numpy as np
import json

from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from minsearch import VectorSearch
from anthropic import Anthropic

from ingest import load_faq_data, build_index
from rag_helper import RAGBase, RAGVector

In [2]:
# load environment variables
# mainly Hugging Face token for downloading model without rate limit and warning
# also loads Anthropic key, but the client can also do this automatically
load_dotenv()

True

In [3]:
# use a small model running locally for getting text embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
# encode a few examples to demonstrate model behavior
# first, encode a question q1
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [5]:
# encode the answer to the q1
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [6]:
# compare the question to the answer using dot product
v1.dot(dv)

np.float32(0.32332394)

In [7]:
# encode and compare an unrelated question
# the similarity with the document should be smaller
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)
v2.dot(dv)

np.float32(0.019730562)

## Embedding the Dataset

In [8]:
# get the course FAQ dataset
documents = load_faq_data()

# check how many documents there are
print(len(documents))

1350


In [9]:
# generate embeddings for the FAQ dataset

# list for containing the embeddings
texts = []

# embedd question and answer together so a query can match both
for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [10]:
# divide the dataset into batches of 50 and encode each batch
batch_size = 50
vectors = []

# encode batches of 50 using the model and add them to the list
# tqdm for progress bar
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

# check result or rather see how many were embedded
# if each documents is embedded, it should be 1350
print(len(vectors))

  0%|          | 0/27 [00:00<?, ?it/s]

1350


It's 1350 vectors, just as the number of documents, so each one should be
embedded now.

In [11]:
# turn the embeddings into a matrix
# rows are documents, columns are dimensions of embeddings
X = np.array(vectors)

# shape should be 1350 x 384
# the small model we use has an output dim of 384
print(X.shape)

(1350, 384)


## Vector Search on the Embeddings

In [12]:
# compute dot product of question embedding against all document embeddings
# dot product equals cosine similarity, because embedding space is L2 normalized
# v1 is the embedding of the question used in the beginning
# scroll up to see it
scores = X.dot(v1)

In [13]:
# get the best match -> highest cosine similarity -> highest score
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.7629408))

In [14]:
# see which Q&A pair it is
# expectation: we asked a literal question from the FAQ -> it will be that one
print("We asked:", q1, "\n") # the question we asked
print("Best match:\n", json.dumps(documents[idx], indent=2))

We asked: Can I still join the course after the start date? 

Best match:
 {
  "course": "data-engineering-zoomcamp",
  "section": "General Course-Related Questions",
  "question": "Course: Can I still join the course after the start date?",
  "answer": "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  "doc_id": "3f1424af17"
}


In [15]:
# get the best 5 results, not just a single one
top5 = np.argsort(-scores)[:5]
top5

array([  2, 625, 907, 538,   7])

In [16]:
# check scores of these top results
scores[top5]

array([0.7629408 , 0.757937  , 0.7192131 , 0.6536311 , 0.56009984],
      dtype=float32)

In [17]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629408
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.757937
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 

## Vector Search with Minsearch

Now we'll use a library for the vector search.
It's just in-memory right now, not persistant in a database.

In [18]:
# initialize a vector search index
vindex = VectorSearch(keyword_fields=["course"])

# fit the index to the documents
vindex.fit(X, documents)

In [19]:
# search the index for a question

# save new question
q3 = "I just discovered the course. Can I still join it?"

# get a text embedding vector for the question
query_vector = model.encode(q3)

# query the index on the vector to return the 5 best matches
results = vindex.search(query_vector, num_results=5)

In [20]:
# look at all top 5 results
print("Top 5 results:")
for result in results:
    print(json.dumps(result, indent=2))
    
# print only the best match
print("\nBest match:\n", results[0]["question"])

Top 5 results:
{
  "course": "llm-zoomcamp",
  "section": "General Course-Related Questions",
  "question": "I just discovered the course. Can I still join?",
  "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions.",
  "doc_id": "74eb249bbf"
}
{
  "course": "machine-learning-zoomcamp",
  "section": "General Course-Related Questions",
  "question": "The course has already started. Can I still join it?",
  "answer": "Yes, you can. Even though you missed the start date, you can register for the course. You won\u2019t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.",
  "doc_id": "41aabbd7c5"
}
{
  "course": "mlops-zoomca

In [21]:
# filter for LLM Zoomcamp only to remove Q&A pairs from other courses
results = vindex.search(
    query_vector=query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [34]:
# look at all top 5 results
print("Top 5 results:")
for result in results:
    print(json.dumps(result, indent=2))
    
# print only the best match
print("\nBest match:\n", results[0]["question"])

Top 5 results:
{
  "course": "llm-zoomcamp",
  "section": "General Course-Related Questions",
  "question": "I just discovered the course. Can I still join?",
  "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions.",
  "doc_id": "74eb249bbf"
}
{
  "course": "llm-zoomcamp",
  "section": "General Course-Related Questions",
  "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
  "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.",
  "doc_id": "69d122f12e"
}
{
  "course": "llm-zoomcamp",
  "section": "General Course-Related Questions",
  "question": "Whe

## Perform RAG with Vector Search

In [23]:
# initialize anthropic client for LLM responses
anthropic_client = Anthropic()

In [24]:
# first, go with text search as baseline
index = build_index(documents)

In [25]:
# initialize a RAG assistant using the text search index
assistant = RAGBase(
    index=index,
    llm_client=anthropic_client,
)

In [26]:
# ask it a question
q4 = "I just found out about the program, can I still sign up?"
response = assistant.rag(q4)
print(response[0])

Yes, you can still sign up for the program! However, there is an important timing consideration:

**If you want to receive a certificate**, you'll need to submit your Capstone project while submissions are still being accepted. So while you can join the course at any time, make sure to complete and submit your project before the submission deadline closes.

Also note that homework assignments are not mandatory to get a certificate—you only need to pass the Capstone project. However, completing homework is recommended for reinforcing the course concepts.


In [29]:
# now switch to vector search for comparison

# initialize a rag assistant
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=anthropic_client,
)

# query it on the same question
response = vector_assistant.rag(q4)
print(response[0])

Yes, you can still sign up! However, there's an important detail to keep in mind:

If you want to **receive a certificate**, you'll need to submit your project while the course is still accepting submissions. Make sure to check the submission deadline so you don't miss it.

You can start learning right away with the course materials, which are pre-recorded and available in the course repository. You don't even need to wait for a confirmation email—you're accepted and can begin submitting homework immediately while the submission form is open.

If you have any questions as you work through the course, feel free to ask on Slack following the [asking questions guidelines](https://datatalks.club/docs/courses/zoomcamp-logistics/asking-questions/).


Basically the same^^ But it's expected, because the text search query was fine.

The query is like this: "I just discovered the course. Can I still join?"

I'll now construct something using entirely different words and test both
assistants again.

In [31]:
q5 = (
    "Found out about this program when it had already started."
    "Is it possible for me to participate in spite of missing its first few weeks?"
)

This new question has basically the same meaning.
It is a bit more precise, but it describes the same issue:
The participant only found the course after it had started already.
Now, they are not sure whether they can still join and participate or not.
Crucially, in the new query there is not a single word also used in the old one.
Not even generic words like "the" or "I".
So, text search should have a hard time finding information about this.
At this point, the RAG is not agentic yet, so my guess is that it won't be able
to answer.
The system prompt should prevent it from providing some generic guess.
A disclaimer: I don't know how realistic this scenario really is.
It might not exactly be super likely that a query has not even a single overlap
in vocabulary with indexed documents in a simple case like this.
But it might still happen, and it will become much more of a problem in more
complex setups.
This is exactly what vector search is supposed to solve here.
The model should actually assing a similar embedding to the new query, because
the meaning is very close.
Consequentially, vector search should still get us a good response here.
Let's try it out!

In [32]:
# text search first
response = assistant.rag(q5)
print(response[0])

Based on the context provided, **yes, you can still participate in the program even if you missed the first few weeks**.

According to the FAQ section on "General Course-Related Questions":

> "Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard."

This indicates that:
1. **Homework is not mandatory** - so missing the first few weeks of homework assignments won't prevent you from participating
2. **The Capstone project is what matters** - you need to pass the Capstone project to get the certificate
3. While homework is recommended for reinforcing concepts, it's not required to continue or complete the program

So you can definitely join now and focus on the remaining material and the Capstone project!


In [33]:
# now vector search
response = vector_assistant.rag(q5)
print(response[0])

# Yes, You Can Still Participate!

Good news! Based on the course guidelines, **yes, you can absolutely join even though you've missed the first few weeks.**

Here are the key points:

## Joining Late
- You can start the course **whenever you want** — the videos and GitHub materials are all available
- There's no need to wait for a specific start date or catch up perfectly
- You don't even need to register to begin learning

## Important for Certificate
If you want to **receive a certificate**, you'll need to:
- **Submit your project while submissions are still being accepted**
- Make sure you meet the project submission deadline (check the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/) for current deadlines)

## How to Get Started
1. Access the course materials from the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)
2. Follow the typical workflow: watch videos → work through notebooks → complete homework
3. Submit your 

Haha ok I did not expect that!
I think there was another question about passing the course, so the text search
assistant was still able to answer the question.
It must have been about the homework and the capstone project.
The RAG assistant seemed to have found the regular answer again.
Let's check both cases manually to confirm!

The RAG class is not limited to just 5 search results, so we have to check more.
The assistant disclosed the section: General course-related questions, so let's
focus on that.
It actually even provided the quote, so let's search for it directly.

In [36]:
# text search index
results = index.search(
    query=q5,
    filter_dict={"course": "llm-zoomcamp"},
)

# look at all results
for result in results:
    print(json.dumps(result, indent=2))

Top 5 results:
{
  "course": "llm-zoomcamp",
  "section": "Capstone Project",
  "question": "Is it a group project?",
  "answer": "No, the capstone is an individual project.\n\nYou can collaborate or discuss a larger idea with other students, but each submitted project must stand on its own. A shared system can work only if it is clearly decomposed into independent projects, where each person has a separate qualifying component and a separate repository.\n\nIf the work cannot be evaluated independently for each participant, it does not satisfy the project requirement.",
  "doc_id": "0fab61eca2"
}
{
  "course": "llm-zoomcamp",
  "section": "General Course-Related Questions",
  "question": "Where is the LLM Zoomcamp Telegram channel?",
  "answer": "The Telegram channel is [https://t.me/llm_zoomcamp](https://t.me/llm_zoomcamp).\n\nUse it for announcements. For technical discussion and questions, use the course Slack channel.",
  "doc_id": "054f3fd58f"
}
{
  "course": "llm-zoomcamp",
  "se

It should have been this one:
```
{
  "course": "llm-zoomcamp",
  "section": "General Course-Related Questions",
  "question": "I missed the first homework - can I still get a certificate?",
  "answer": "Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.",
  "doc_id": "9f689c185f"
}
```

This helped it to derive that only the capstone project is mandatory.
Let's first see if the vector search index still returns the original question
and then further modify the query so the text search won't find anything - just
for fun.

In [37]:
# vector
results = vindex.search(
    query_vector=query_vector,
    filter_dict={"course": "llm-zoomcamp"},
)

# look at all results
print("Top 5 results:")
for result in results:
    print(json.dumps(result, indent=2))

Top 5 results:
{
  "course": "llm-zoomcamp",
  "section": "General Course-Related Questions",
  "question": "I just discovered the course. Can I still join?",
  "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions.",
  "doc_id": "74eb249bbf"
}
{
  "course": "llm-zoomcamp",
  "section": "General Course-Related Questions",
  "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
  "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.",
  "doc_id": "69d122f12e"
}
{
  "course": "llm-zoomcamp",
  "section": "General Course-Related Questions",
  "question": "Whe

As expected, the question is still the first match.
Briefly compare embeddings, then move on to modifying the query.
The embeddings should be similar.

In [38]:
# original question was already embedded to v1
# encode q5 on the fly
v1.dot(model.encode(q5))

np.float32(0.51644725)

That's not too close, but quite close.
See how close the others are in comparison.

In [53]:
# get llm-zoomcamp documents
llmzc_documents = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]

# get scores for LLM Zoomcamp FAQ
llmzc_texts = []
for doc in llmzc_documents:
    text = doc["question"] + " " + doc["answer"]
    llmzc_texts.append(text)

batch_size = 50
llmzc_vectors = []

for i in tqdm(range(0, len(llmzc_texts), batch_size)):
    batch = llmzc_texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    llmzc_vectors.extend(batch_vectors)

# check if there are now 85 docs
print(len(llmzc_vectors))

# get matrix
llmzc_X = np.array(llmzc_vectors)

  0%|          | 0/2 [00:00<?, ?it/s]

85


In [54]:
# get other close matches
llmzc_scores = llmzc_X.dot(model.encode(q5))
llmzc_top5 = np.argsort(-llmzc_scores)[:5]
llmzc_scores[llmzc_top5]

array([0.45895496, 0.39008707, 0.30043042, 0.2917397 , 0.2908018 ],
      dtype=float32)

In [55]:
# see which documents these are
for idx in llmzc_top5:
    print(llmzc_scores[idx])
    print(llmzc_documents[idx])
    print()

0.45895496
{'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.', 'doc_id': '74eb249bbf'}

0.39008707
{'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?', 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.", 'doc_id': '977bf7786c'}

0.30043042
{'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Are there live sessions or office hours for each module?', 'answer': 'There are no separate live sessions for

Okay makes sense.
The match is the closest, but the other ones also are related to registration,
live sessions etc.

This one 'How should I start the course and follow the weekly workflow?' also
contains info that the participants can start whenever they want.

Let's make sure that text search also won't find that one and hope there is not
even more :D
If it still gets it, I will move on, because this is essentially just for fun
and I need to progress with the course and other important projects, too.

For recap: Text search mustn't find this one:
'I missed the first homework - can I still get a certificate?'

Okay wait! I just noticed that actually the document consists of query AND
answer, so it will be very hard to fully exclude the correct answers.
I will give this one single try, and if it doesn't work, I'll really move on!

That means this may not appear:
"Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard."

And also not this: "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."

This takes too much time right now, so I'll use Claude to generate a prompt not
containing any of the keywords instead of doing it myself.

In [ ]:
q6 = (
    "Found out about this program when it had already started. "
    "Is it possible for me to access the material in spite of arriving late?"
)

The above query q6 should not match the two Q&A pairs helping the text search.
If there are any further ones in there, then the model will still be able to
answer, but at that point my little exercise will become pointless.
Last try now!

Also, I just learned in the convo with Claude that words like "to", "is" etc. are
regarded as "stop words" and are typically filtered out, so they won't generate
any matches.
Makes total sense, because they don't provide any semantical meaning.
I hope minsearch works like this, too.

Vector first this time:

In [57]:
# vector
response = vector_assistant.rag(q5)
print(response[0])

# Yes, You Can Still Participate!

According to the course guidelines, **yes, you can absolutely join the program even if you've missed the first few weeks.**

Here are the key points:

1. **You can start anytime**: The videos and GitHub materials are available whenever you want to begin. You don't need to wait for a specific start date.

2. **Certificate requirement**: If you want to receive a certificate, you'll need to submit your project while the course is still accepting submissions. So timing matters mainly for the final project deadline.

3. **No registration barrier**: You don't even need a confirmation email to participate. You can simply start learning and submitting homework right away.

4. **Self-paced workflow**: The typical workflow is:
   - Watch the lesson videos
   - Work through the lesson notebooks/code
   - Complete homework assignments
   - Submit before deadlines

**My recommendation**: Check the [course management platform](https://courses.datatalks.club/llm-zoo

As expected, this works, because the semantics are identified and matched.
Now let's see if texts will find any other stuff in the FAQ I am not aware of
again :D

In [58]:
# text
response = assistant.rag(q5)
print(response[0])

# Answer

Yes, you can participate in the program even if you've missed the first few weeks!

According to the course information, **homework is not mandatory**. While it is recommended for reinforcing concepts and the points count towards your leaderboard rank, missing homework assignments won't prevent you from continuing.

The key requirement for getting a certificate is to **pass the Capstone project**. Since the capstone is an individual project that you'll work on, you should be able to join and complete it successfully even if you've missed the earlier modules.

My recommendation would be to:
1. Join the program now
2. Review the earlier module materials at your own pace if needed
3. Focus on completing the Capstone project to earn your certificate


Yeah well this didn't work so well.
I was not able to fully exclude all information from text search.

However, this still demonstrated a few things.
First of all, I was indeed able to exclude the original question from the
search results.
While this is neither new nor surprising, it was still cool to witness that
text search was not able to use any level of semantic understanding.

More importantly, however, I'd say that this actually demonstrated the
robustness of a simple text search approach given a well structured knowledge
base with some degree of redundance or rather interconnectedness of answers.
I was not able to confuse the simple baseline assistant.

Still, vector search proved to be the stronger approach, because each time it
found the exact right match for the question.

Next, I'll make the vector search index persistant using sqlitesearch.